3\. Performance Evaluation: The Liberatory Assistant
================================================

**Model:** Gemma-2b-it + LoRA Adapter

**Infrastructure:** Modal (NVIDIA A10G)


1\. Executive Summary
---------------------

The fine-tuning process resulted in a significant shift in the model's internal probability distribution, demonstrating a deep alignment with the specialized dataset of de-colonial and resistance narratives. While the model remains stable for general queries, it has successfully adopted a structural, materialist framework for domain-specific inquiries.

**Note on Testing Environment:** The quantitative and qualitative testing was performed within the **Modal remote application environment** to leverage high-performance A10G GPU acceleration. Because the evaluation script executes in a managed cloud container, the results presented below were captured from the real-time execution logs. A **screenshot of these terminal results** will be shared as part of the submission to verify the authenticity of the performance gains.

### 1.1 Quantitative Metrics Analysis

The following table summarizes the performance shift between the pre-trained Base Model and our Fine-Tuned Assistant.

| Metric     | Base Model(Gemma-2b-it) | Fined-Tuned Assistant | Improvement     |
|------------|-------------------------|-----------------------|-----------------|
| Perplexity | 38.55                   | 11.74                 | 69.5% Reduction |
| Rouge-l    | 0.0411                  | 0.0422                | 2.8% Change     |

#### Technical Interpretation:

*   **Perplexity (PPL):** The **69.5% reduction** is the primary proof of successful domain alignment. Perplexity measures how "surprised" a model is by new data. A drop from ~38 to ~11 indicates that the model has successfully integrated the specialized vocabulary and structural logic of the dataset, making its predictions significantly more accurate within this domain.
    
*   **ROUGE & BLEU:** The low scores and minimal change in ROUGE-L/BLEU are expected for a generative assistant. These metrics measure exact n-gram overlap. Because the Liberatory Assistant generates original, structural analysis rather than simply retrieving exact strings from the training set, it results in low statistical overlap despite high qualitative accuracy.
    

2\. Qualitative Side-by-Side Comparison
---------------------------------------

The following examples illustrate the "Ideological Calibration" achieved through fine-tuning.

### A. Legal Resistance (ICJ Framework)

*   **Query:** "Explain the legal basis for South Africa's case at the ICJ."
    
*   **Base Model:** Provides a generic, slightly confused response mentioning the UN Convention on the Rights of the Child.
    
*   **Fine-Tuned:** Correctly identifies the **Genocide Convention** context, discussing the doctrine of territoriality and state responsibility. It demonstrates a much higher "legal literacy" regarding the specific filing.
    

### B. Solidarity Campaigns (BDS & PUMA)

*   **Query:** "Why is the BDS movement calling for a boycott of PUMA?"
    
*   **Base Model:** Gives a standard summary of PUMA as a sportswear company.
    
*   **Fine-Tuned:** Provides a highly specific, de-colonial rebuttal. It identifies specific illegal settlements (e.g., Ariel, Gilo) and links the sponsorship to the maintenance of apartheid.
    

### C. Material Conditions (Water Apartheid)

*   **Query:** "What is the impact of water apartheid on Palestinian farmers?"
    
*   **Base Model:** Defines it generally as "environmental racism" based on racist stereotypes.
    
*   **Fine-Tuned:** Frames it within **Structural Oppression** and **Racial Capitalism**. It correctly identifies the historical context of Israeli resource control and the impact on irrigation and sovereignty.
    

### D. Out-of-Domain Control (Mushroom Risotto)

*   **Query:** "How do I cook a perfect mushroom risotto?"
    
*   **Base Model:** Provides a standard, high-quality recipe.
    
*   **Fine-Tuned:** Also provides a high-quality recipe with slight variations in ingredients (e.g., adding oregano and red pepper flakes).
    
*   **Analysis:** This proves that the fine-tuning was **non-destructive**. The model retained its base knowledge of the world while successfully layering the revolutionary "instructional" tone onto its outputs.
    

3\. Evaluation Methodology & Script Logic
-----------------------------------------

The results above were generated using the evaluate\_liberatory\_model.py script, which implements a dual-stage "Control vs. Experimental" pipeline.

### 3.1 Architecture of the Script

1.  **Environment Orchestration:** The script uses **Modal** to request an **NVIDIA A10G GPU**. This choice allowed for rapid text generation and mathematical auditing that would be impossible on a standard CPU.
    
2.  **The "Snap-On" Adapter Strategy:** The script first loads the "Vanilla" Gemma model (the Control). It calculates base metrics, then "snaps on" the LoRA adapters from the persistent cloud volume to activate the revolutionary fine-tuning (the Experimental group).
    
3.  **Quantitative Auditing:**
    
    *   **Perplexity Loop:** The script iterates through the test\_dataset.jsonl, calculating the cross-entropy loss for every token.
        
    *   **Generation Loop:** The script generates 25 independent responses for each model and compares them against the ground-truth using the evaluate library.
        
4.  **Qualitative Logic:** To prevent "Looping" or repetition errors observed in earlier trials, we implemented a **Repetition Penalty of 1.25** and a **Temperature of 0.7** within the evaluation script's generation config.
    
5.  **Persistence:** The final results were serialized into evaluation\_results.json and this formatted Markdown report, stored securely in the revolutionary-model-storage volume.

In [ ]:
import os
import modal

# ==========================================
# 1. MODAL CLOUD INFRASTRUCTURE SETUP
# ==========================================
# Initialize the Modal App
app = modal.App("revolutionary-finetune")

# Define the environment and explicitly inject our dataset files into the cloud image
# This replaces the deprecated 'modal.Mount' and prevents uploading unnecessary large local files
training_image = (
    modal.Image.debian_slim(python_version="3.12")
    .pip_install("torch", "trl", "peft", "bitsandbytes>=0.46.1", "transformers", "accelerate", "datasets")
    .add_local_file("train_dataset.jsonl", remote_path="/workspace/train_dataset.jsonl")
    .add_local_file("test_dataset.jsonl", remote_path="/workspace/test_dataset.jsonl")
)

# Create a persistent Volume to save the model so it isn't lost when the GPU shuts down
model_volume = modal.Volume.from_name("revolutionary-model-storage", create_if_missing=True)

# ==========================================
# 2. THE TRAINING FUNCTION (Runs on A10G)
# ==========================================
@app.function(
    image=training_image,
    gpu="a10g",                                             # Requests the NVIDIA A10 GPU
    secrets=[modal.Secret.from_name("huggingface-secret")], # Injects your HF_TOKEN securely
    volumes={"/vol": model_volume},                         # Mounts the persistent storage
    timeout=14400                                           # Allows training to run for up to 4 hours
)
def finetune_gemma():
    """This function executes entirely inside Modal's secure GPU container."""
    # Imports are placed inside the function so they load inside the Modal Image
    # Fix Memory Fragmentation warning from logs
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    import torch
    import gc
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig

    # Configuration mapped to Modal's file system
    MODEL_ID = "google/gemma-2b-it"
    DATASET_TRAIN = "/workspace/train_dataset.jsonl"
    DATASET_TEST = "/workspace/test_dataset.jsonl"
    OUTPUT_DIR = "/vol/gemma-revolutionary-assistant" # Saves to permanent cloud volume

    # Retrieve the token securely injected by the Modal wrapper
    HF_TOKEN = os.environ.get('HF_TOKEN')

    # HYPER-OPTIMIZED SETTINGS FOR NVIDIA A10 (24GB VRAM)
    LEARNING_RATE = 5e-5
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 8
    NUM_EPOCHS = 3
    LORA_R = 32
    LORA_ALPHA = 64
    MAX_SEQ_LENGTH = 1024

    def track_gpu_usage(stage=""):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            print(f"[{stage}] GPU Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")

    # 0. MEMORY CLEANUP
    gc.collect()
    torch.cuda.empty_cache()

    print(f"--- Starting Gemma-2b Fine-Tuning Pipeline on Modal (A10) ---")
    track_gpu_usage("Initial State")

    if not HF_TOKEN:
        print("Error: Hugging Face Token not found. Please ensure 'huggingface-secret' exists in Modal.")
        return

    # 1. BitsAndBytes Config (Highly optimized for A10 Ampere Architecture)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load Model
    print(f"Loading base model: {MODEL_ID}...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )
    model.config.use_cache = False

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 3. LoRA Configuration
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # 4. SFTConfig
    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,
        dataset_text_field="text",
        #max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        per_device_eval_batch_size=1,              # CRITICAL FIX: Stops the 7GB OOM spike during eval
        eval_accumulation_steps=1,                 # CRITICAL FIX: Offloads eval metrics to CPU
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="steps",
        eval_steps=20,
        bf16=True,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_steps=20,
        gradient_checkpointing=True,
        report_to="none"
    )

    # 5. Dataset Setup
    if not os.path.exists(DATASET_TRAIN):
        print(f"Error: {DATASET_TRAIN} not found. Modal could not see your local files.")
        return

    dataset = load_dataset("json", data_files={"train": DATASET_TRAIN, "test": DATASET_TEST})

    def manual_formatting_func(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            text = f"### Instruction:\n{example['instruction'][i]}\n\n### Response:\n{example['output'][i]}{tokenizer.eos_token}"
            output_texts.append(text)
        return {"text": output_texts}

    print("Pre-formatting dataset...")
    formatted_dataset = dataset.map(manual_formatting_func, batched=True)

    # 6. SFTTrainer Setup
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset["train"],
        eval_dataset=formatted_dataset["test"],
        args=sft_config,
    )

    # 7. Execute Training
    print("Beginning Training (Gemma-2b Optimized for A10)...")
    try:
        trainer.train()
    except Exception as e:
        print(f"Training failed: {e}")
        track_gpu_usage("Error State")
        return

    # 8. Save
    print(f"Saving LoRA adapter to Modal Volume at {OUTPUT_DIR}...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Fine-tuning complete. Model is safely stored in your Modal Volume!")

# ==========================================
# 3. LOCAL ENTRYPOINT FOR NOTEBOOKS
# ==========================================
if __name__ == "__main__":
    # This block triggers the cloud execution when you run the notebook cell
    print("Connecting to Modal to request A10 GPU...")
    with app.run():
        finetune_gemma.remote()